<!-- Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. -->

# MJX 02 — Rendering, Cameras & Contacts

Goal: master offscreen rendering control.

- **`mujoco.Renderer`**: render any camera to an RGB array.
- **Named cameras** defined in the MJCF vs the default free camera.
- **`MjvOption.geomgroup`**: MuJoCo geoms belong to groups; **visual** meshes and **collision** shapes are usually in different groups. Showing the wrong group is exactly why robots can look wrong (e.g. a green collision shape covering a white visual mesh).
- **Contact visualization**: enable contact-point flags to see where bodies touch.

Everything runs headless on the GPU via EGL.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import matplotlib.pyplot as plt
import mujoco
import imageio

In [ ]:
# A scene with: two named cameras, a body carrying BOTH a visual geom (group 1)
# and a collision geom (group 0), and a falling box to generate contacts.
MJCF = """
<mujoco model="render_demo">
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="cam_front" pos="0 -2.2 1.0" xyaxes="1 0 0 0 0.5 1"/>
    <camera name="cam_side" pos="2.2 0 1.0" xyaxes="0 1 0 -0.5 0 1"/>
    <geom name="floor" type="plane" size="3 3 0.1" rgba="0.8 0.9 0.8 1"/>
    <body name="dual" pos="-0.5 0 0.5">
      <geom name="vis" type="box" size="0.1 0.1 0.1" group="1" rgba="0.9 0.2 0.2 1"/>
      <geom name="col" type="sphere" size="0.17" group="0" rgba="0.2 0.2 0.9 0.4"/>
    </body>
    <body name="faller" pos="0.6 0 1.2">
      <freejoint/>
      <geom name="fall" type="box" size="0.12 0.12 0.12" rgba="0.2 0.7 0.3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(MJCF)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
renderer = mujoco.Renderer(model, height=320, width=320)

In [ ]:
# Named cameras: render the same scene from two viewpoints.
imgs = {}
for cam in ["cam_front", "cam_side"]:
    renderer.update_scene(data, camera=cam)
    imgs[cam] = renderer.render()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (name, img) in zip(axes, imgs.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Geom groups: the body 'dual' has a red visual box (group 1) and a translucent
# blue collision sphere (group 0). Toggling groups changes what you see.
opt_all = mujoco.MjvOption()  # default: groups 0,1,2 visible

opt_visual = mujoco.MjvOption()
opt_visual.geomgroup[:] = 0
opt_visual.geomgroup[1] = 1  # visual geoms only

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (title, opt) in zip(axes, [("all groups (visual+collision)", opt_all), ("visual group only", opt_visual)]):
    renderer.update_scene(data, camera="cam_front", scene_option=opt)
    ax.imshow(renderer.render()); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Contacts: let the green box fall onto the floor and visualize contact points.
os.makedirs("output/videos", exist_ok=True)
contact_opt = mujoco.MjvOption()
contact_opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
contact_opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True

mujoco.mj_resetData(model, data)
frames = []
for i in range(500):
    mujoco.mj_step(model, data)
    if i % 4 == 0:
        renderer.update_scene(data, camera="cam_front", scene_option=contact_opt)
        frames.append(renderer.render())

video_path = "output/videos/mjx02_contacts.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Saved {len(frames)} frames to {video_path}")

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx02_contacts.mp4")